# Programming Question Retrieval
## Phase 2 — NLP Text Preprocessing

**Purpose:** transform the prepared Stack Overflow question documents into consistent tokens for the next Information Retrieval stage (Inverted Index).

**Input:** `documents.csv` from Phase 1. Each row represents one question. The text source is `Document`, combining `Title`, cleaned `Body`, and `Tags`.

**Output:** `processed_documents.csv.gz`, preserving the original columns and adding `Tokens` and `ProcessedDocument`.

This notebook is tailored to programming questions. It keeps technical forms such as `c++`, `c#`, `.net`, `node.js`, `react-native`, and `snake_case`. It deliberately preserves negation words such as `not`, `no`, and `without`.

## 1. Imports

In [127]:
import os
import re
import json
import sys
import pandas as pd
from IPython.display import display

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

Python: 3.13.18
Pandas: 2.3.3


## 2. Initialize WordNet lemmatization

NLTK's WordNet lemmatizer requires the WordNet lexical database. The download is attempted only if the resource is missing. If the environment has no internet access and the resource is unavailable, this cell stops rather than silently skipping lemmatization.

In [128]:
import nltk
from nltk.stem import WordNetLemmatizer

try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet", quiet=True)

try:
    nltk.data.find("corpora/omw-1.4")
except LookupError:
    nltk.download("omw-1.4", quiet=True)

lemmatizer = WordNetLemmatizer()
_ = lemmatizer.lemmatize("libraries")
print("NLTK and WordNet are ready.")

[nltk_data] Error loading wordnet: <urlopen error [Errno -3] Temporary
[nltk_data]     failure in name resolution>


NLTK and WordNet are ready.


[nltk_data] Error loading omw-1.4: <urlopen error [Errno -3] Temporary
[nltk_data]     failure in name resolution>


## 3. Load the Phase 1 document collection

Set `INPUT_PATH` to the actual location of your Phase 1 file. Pandas reads gzip-compressed CSV files directly when the filename ends in `.gz`.

In [129]:
import glob
files = glob.glob("/kaggle/input/**/*", recursive=True)
files = [f for f in files if os.path.isfile(f) and "documents" in os.path.basename(f).lower()]

if not files:
    raise FileNotFoundError("/kaggle/input/datasets/emanmajdoob/documents-phase1/documents.csv")

INPUT_PATH = files[0]
print("Using file:", INPUT_PATH)

documents_df = pd.read_csv(INPUT_PATH, compression="infer")

print("Dataset loaded successfully.")
print("Shape:", documents_df.shape)
print("Columns:", documents_df.columns.tolist())

display(documents_df.head(3))

Using file: /kaggle/input/datasets/emanmajdoob/documents-phase1/documents.csv
Dataset loaded successfully.
Shape: (44997, 7)
Columns: ['Id', 'Title', 'Body', 'Tags', 'CreationDate', 'Y', 'Document']


,Id,Title,Body,Tags,CreationDate,Y,Document
0,34552656,Java: Repeat Task Every Random Seconds,I'm already familiar with repeating tasks ever...,java repeat,2016-01-01 00:21:59,LQ_CLOSE,Java: Repeat Task Every Random Seconds I'm alr...
1,34553034,Why are Java Optionals immutable?,I'd like to understand why Java 8 Optionals we...,java optional,2016-01-01 02:03:20,HQ,Why are Java Optionals immutable? I'd like to ...
2,34553174,Text Overlay Image with Darkened Opacity React...,I am attempting to overlay a title over an ima...,javascript image overlay react-native opacity,2016-01-01 02:48:24,HQ,Text Overlay Image with Darkened Opacity React...


## 4. Validate the input data

We check required columns, duplicate IDs, missing values, and empty documents. Invalid rows are excluded only from the NLP working copy; the loaded source DataFrame remains unchanged.

In [130]:
required_columns = {"Id", "Document"}
missing_columns = required_columns - set(documents_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Missing values:")
display(documents_df[["Id", "Document"]].isna().sum())
print("Duplicate IDs:", documents_df["Id"].duplicated().sum())

empty_mask = documents_df["Document"].fillna("").astype(str).str.strip().eq("")
print("Empty documents:", int(empty_mask.sum()))

nlp_df = documents_df.loc[~empty_mask].copy().reset_index(drop=True)
nlp_df["Document"] = nlp_df["Document"].fillna("").astype(str)
print("Documents available for NLP:", len(nlp_df))

Missing values:


Id          0
Document    0
dtype: int64

Duplicate IDs: 0
Empty documents: 0
Documents available for NLP: 44997


## 5. Lowercasing

Lowercasing makes text case-insensitive for retrieval: `Python`, `PYTHON`, and `python` become the same form. The original `Document` is preserved. This is a practical baseline for text retrieval, although source code itself can be case-sensitive.

In [131]:
nlp_df["LowercasedDocument"] = nlp_df["Document"].str.lower()
display(nlp_df[["Id", "Document", "LowercasedDocument"]].head(5))

,Id,Document,LowercasedDocument
0,34552656,Java: Repeat Task Every Random Seconds I'm alr...,java: repeat task every random seconds i'm alr...
1,34553034,Why are Java Optionals immutable? I'd like to ...,why are java optionals immutable? i'd like to ...
2,34553174,Text Overlay Image with Darkened Opacity React...,text overlay image with darkened opacity react...
3,34553318,Why ternary operator in swift is so picky? The...,why ternary operator in swift is so picky? the...
4,34553755,hide/show fab with scale animation I'm using c...,hide/show fab with scale animation i'm using c...


## 6. Programming-aware tokenization

A token is a text unit that the retrieval system can index. A plain whitespace split can damage technical terms. The pattern below keeps common forms such as `c++`, `c#`, `f#`, `.net`, `node.js`, `react-native`, `snake_case`, and identifiers containing digits.

This is a practical baseline tokenizer, not a full programming-language lexer. Inspect the examples before accepting it.

In [132]:
TOKEN_PATTERN = re.compile(
    r"(?ix)c\+\+|c\#|f\#|\.net|[a-z0-9]+(?:[._+-][a-z0-9]+)*"
)

def tokenize(text):
    # Return lowercase tokens while retaining selected programming terms.
    if not isinstance(text, str):
        return []
    return TOKEN_PATTERN.findall(text.lower())

nlp_df["RawTokens"] = nlp_df["LowercasedDocument"].apply(tokenize)
display(nlp_df[["Id", "LowercasedDocument", "RawTokens"]].head(5))

,Id,LowercasedDocument,RawTokens
0,34552656,java: repeat task every random seconds i'm alr...,"[java, repeat, task, every, random, seconds, i..."
1,34553034,why are java optionals immutable? i'd like to ...,"[why, are, java, optionals, immutable, i, d, l..."
2,34553174,text overlay image with darkened opacity react...,"[text, overlay, image, with, darkened, opacity..."
3,34553318,why ternary operator in swift is so picky? the...,"[why, ternary, operator, in, swift, is, so, pi..."
4,34553755,hide/show fab with scale animation i'm using c...,"[hide, show, fab, with, scale, animation, i, m..."


### 6.1 Inspect tokenizer behavior

Run this small test to check whether important programming terms remain intact.

In [133]:
tokenizer_examples = [
    "C++ and C# work with .NET",
    "Use node.js with react-native",
    "A variable_name and python3 identifier",
    "How to use pandas.DataFrame?",
    "The value is not equal to 0"
]

for example in tokenizer_examples:
    print("TEXT :", example)
    print("TOKENS:", tokenize(example))
    print("-" * 65)

TEXT : C++ and C# work with .NET
TOKENS: ['c++', 'and', 'c#', 'work', 'with', '.net']
-----------------------------------------------------------------
TEXT : Use node.js with react-native
TOKENS: ['use', 'node.js', 'with', 'react-native']
-----------------------------------------------------------------
TEXT : A variable_name and python3 identifier
TOKENS: ['a', 'variable_name', 'and', 'python3', 'identifier']
-----------------------------------------------------------------
TEXT : How to use pandas.DataFrame?
TOKENS: ['how', 'to', 'use', 'pandas.dataframe']
-----------------------------------------------------------------
TEXT : The value is not equal to 0
TOKENS: ['the', 'value', 'is', 'not', 'equal', 'to', '0']
-----------------------------------------------------------------


## 7. Conservative stopword removal

Removing a standard English stopword list blindly can damage programming questions. Removing `not`, for example, changes the meaning of “does not work”. We therefore use a small conservative list and explicitly protect negation and common programming keywords. This is a baseline decision to evaluate with sample queries.

In [134]:
STOPWORDS = {
    "a", "an", "the",
    "am", "is", "are", "was", "were", "be", "been", "being",
    "do", "does", "did",
    "this", "that", "these", "those",
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "they", "them",
    "it", "its",
    "please", "thanks", "thank",
    "very", "just", "also"
}

PROTECTED_TERMS = {
    "not", "no", "nor", "without",
    "if", "else", "for", "while", "in", "on", "as", "from", "with",
    "class", "return", "import", "true", "false", "null", "none"
}
STOPWORDS -= PROTECTED_TERMS

def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOPWORDS]

nlp_df["FilteredTokens"] = nlp_df["RawTokens"].apply(remove_stopwords)
display(nlp_df[["Id", "RawTokens", "FilteredTokens"]].head(5))
print("Number of stopwords:", len(STOPWORDS))
print("Protected terms:", sorted(PROTECTED_TERMS))

,Id,RawTokens,FilteredTokens
0,34552656,"[java, repeat, task, every, random, seconds, i...","[java, repeat, task, every, random, seconds, m..."
1,34553034,"[why, are, java, optionals, immutable, i, d, l...","[why, java, optionals, immutable, d, like, to,..."
2,34553174,"[text, overlay, image, with, darkened, opacity...","[text, overlay, image, with, darkened, opacity..."
3,34553318,"[why, ternary, operator, in, swift, is, so, pi...","[why, ternary, operator, in, swift, so, picky,..."
4,34553755,"[hide, show, fab, with, scale, animation, i, m...","[hide, show, fab, with, scale, animation, m, u..."


Number of stopwords: 37
Protected terms: ['as', 'class', 'else', 'false', 'for', 'from', 'if', 'import', 'in', 'no', 'none', 'nor', 'not', 'null', 'on', 'return', 'true', 'while', 'with', 'without']


## 8. Lemmatization

Lemmatization maps inflected English words to a dictionary base form. For example, the noun `libraries` becomes `library`.

We use WordNet's default noun lemmatization as a lightweight baseline. To avoid changing programming-specific terms, only alphabetic tokens are lemmatized; tokens containing digits, dots, plus signs, hashes, hyphens, or underscores are preserved.

**Limitation:** WordNet's default lemmatizer assumes noun part-of-speech. It is not a context-aware POS tagger and may not reduce every verb form (for example, `running`). This limitation should be documented if a more advanced lemmatizer is not added.

In [135]:

def lemmatize_token(token):

    # Preserve programming terms and protected words
    if not token.isalpha() or token in PROTECTED_TERMS:
        return token

    return lemmatizer.lemmatize(token)


def lemmatize_tokens(tokens):

    lemmas = [lemmatize_token(token) for token in tokens]

    # Remove stopwords again after lemmatization
    return [
        token for token in lemmas
        if token not in STOPWORDS
    ]


nlp_df["Tokens"] = nlp_df["FilteredTokens"].apply(
    lemmatize_tokens
)

nlp_df["ProcessedDocument"] = nlp_df["Tokens"].apply(
    lambda tokens: " ".join(tokens)
)

print("Lemmatization updated successfully.")


Lemmatization updated successfully.


## 9. Build the final processed text

`Tokens` is the primary output for the next stage. `ProcessedDocument` is a convenient string version for inspection. The original `Document` remains unchanged. The quality label `Y` is metadata and must not be included in searchable text.

In [136]:
nlp_df["ProcessedDocument"] = nlp_df["Tokens"].apply(lambda tokens: " ".join(tokens))
processed_df = nlp_df.copy()
display(processed_df[["Id", "Document", "Tokens", "ProcessedDocument"]].head(5))

,Id,Document,Tokens,ProcessedDocument
0,34552656,Java: Repeat Task Every Random Seconds I'm alr...,"[java, repeat, task, every, random, second, m,...",java repeat task every random second m already...
1,34553034,Why are Java Optionals immutable? I'd like to ...,"[why, java, optionals, immutable, d, like, to,...",why java optionals immutable d like to underst...
2,34553174,Text Overlay Image with Darkened Opacity React...,"[text, overlay, image, with, darkened, opacity...",text overlay image with darkened opacity react...
3,34553318,Why ternary operator in swift is so picky? The...,"[why, ternary, operator, in, swift, so, picky,...",why ternary operator in swift so picky questio...
4,34553755,hide/show fab with scale animation I'm using c...,"[hide, show, fab, with, scale, animation, m, u...",hide show fab with scale animation m using cus...


## 10. Before-and-after inspection

Inspect real records. Check technical terms, negation, removed words, and unusually short token lists.

In [137]:
sample_columns = ["Id", "Title", "Tags", "Document", "Tokens", "ProcessedDocument"]
sample_columns = [c for c in sample_columns if c in processed_df.columns]

with pd.option_context("display.max_colwidth", 220):
    display(processed_df[sample_columns].sample(
        min(10, len(processed_df)), random_state=42
    ))

,Id,Title,Tags,Document,Tokens,ProcessedDocument
23141,45393123,Adding calculated column in Pandas,python pandas,"Adding calculated column in Pandas I have a dataframe with 10 columns. I want to add a new column 'age_bmi' which should be a calculated column multiplying 'age' * 'bmi'. age is an INT, bmi is a FLOAT. That then crea...","[adding, calculated, column, in, panda, have, dataframe, with, 10, column, want, to, add, new, column, age_bmi, which, should, calculated, column, multiplying, age, bmi, age, int, bmi, float, then, creates, new, data...",adding calculated column in panda have dataframe with 10 column want to add new column age_bmi which should calculated column multiplying age bmi age int bmi float then creates new dataframe with 11 column something ...
9908,39251291,Is it possible to filter options requests in chrome inspector?,google-chrome-devtools,Is it possible to filter options requests in chrome inspector? I'm building a web application which uses CORS to communicate with it's server. This means that every request is preceded by one with OPTIONS method. Is ...,"[possible, to, filter, option, request, in, chrome, inspector, m, building, web, application, which, us, cors, to, communicate, with, s, server, mean, every, request, preceded, by, one, with, option, method, possible...",possible to filter option request in chrome inspector m building web application which us cors to communicate with s server mean every request preceded by one with option method possible to filter out in inspector ma...
8899,38759428,When is `new Error()` better than `Error()`?,javascript error-handling idiomatic,"When is `new Error()` better than `Error()`? The ES5 language spec clearly states that Error(foo) does the same thing as new Error(foo) . But I notice that in the wild, the longer new Error(foo) form is much more com...","[when, new, error, better, than, error, es5, language, spec, clearly, state, error, foo, same, thing, as, new, error, foo, but, notice, in, wild, longer, new, error, foo, form, much, more, common, there, some, reason...",when new error better than error es5 language spec clearly state error foo same thing as new error foo but notice in wild longer new error foo form much more common there some reason for there any situation where usi...
13757,40768459,PHP/MySQLi Contact Form Not Working with $_POST,php html mysqli prepared-statement,PHP/MySQLi Contact Form Not Working with $_POST So my form seems to work when i put test values in the php file like so: $name = 'name'; $email = 'test@gmail.com'; $telephone = '123456789'; $message = 'test message';...,"[php, mysqli, contact, form, not, working, with, post, so, form, seems, to, work, when, put, test, value, in, php, file, like, so, name, name, email, test, gmail.com, telephone, 123456789, message, test, message, how...",php mysqli contact form not working with post so form seems to work when put test value in php file like so name name email test gmail.com telephone 123456789 message test message however when replace test value with...
39842,55723772,How to run react-native run-ios with specific target,react-native react-native-cli,How to run react-native run-ios with specific target Is there a way to run a specific target with command react-native run-ios for Android I'm using following react-native run-android --variant=targetRelease react-na...,"[how, to, run, react-native, run-ios, with, specific, target, there, way, to, run, specific, target, with, command, react-native, run-ios, for, android, m, using, following, react-native, run-android, variant, target...",how to run react-native run-ios with specific target there way to run specific target with command react-native run-ios for android m using following react-native run-android variant targetrelease react-native react-...
18494,42950561,How can I get the raw request body in a Google Cloud Function?,javascript firebase google-cloud-functions,How can I get the raw request body in a Google Cloud Function

## 11. Processing statistics and quality checks

These statistics describe preprocessing output. They do not measure retrieval effectiveness; retrieval quality must be evaluated later using representative queries and relevance judgments.

In [138]:
processed_df["TokenCount"] = processed_df["Tokens"].apply(len)

total_tokens = int(processed_df["TokenCount"].sum())
unique_tokens = {
    token
    for token_list in processed_df["Tokens"]
    for token in token_list
}

print("NLP processing summary")
print("-" * 45)
print("Input documents:", len(documents_df))
print("Documents processed:", len(processed_df))
print("Documents with tokens:", int((processed_df["TokenCount"] > 0).sum()))
print("Documents with zero tokens:", int((processed_df["TokenCount"] == 0).sum()))
print("Total tokens:", total_tokens)
print("Average tokens per document:", round(processed_df["TokenCount"].mean(), 2))
print("Unique tokens:", len(unique_tokens))

display(processed_df.loc[
    processed_df["TokenCount"] == 0, ["Id", "Title", "Tags"]
].head(10))

NLP processing summary
---------------------------------------------
Input documents: 44997
Documents processed: 44997
Documents with tokens: 44997
Documents with zero tokens: 0
Total tokens: 4687293
Average tokens per document: 104.17
Unique tokens: 273076


,Id,Title,Tags


## 12. Save the NLP output

The output is a compressed CSV. Token lists are serialized as JSON arrays (for example, `["python", "list", "sort"]`) so they can be reconstructed reliably in the next notebook. Original columns are retained.

In [139]:
save_df = processed_df.copy()
save_df["Tokens"] = save_df["Tokens"].apply(
    lambda tokens: json.dumps(tokens, ensure_ascii=False)
)

OUTPUT_PATH = "processed_documents.csv.gz"
save_df.to_csv(OUTPUT_PATH, index=False, compression="gzip")

size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print("Saved:", OUTPUT_PATH)
print("Rows:", len(save_df))
print(f"File size: {size_mb:.2f} MB")

Saved: processed_documents.csv.gz
Rows: 44997
File size: 48.75 MB


## 13. Reload test

Verify that the saved file can be reopened and that token lists can be reconstructed.

In [140]:
reloaded_df = pd.read_csv(OUTPUT_PATH, compression="gzip", low_memory=False)
reloaded_df["Tokens"] = reloaded_df["Tokens"].apply(json.loads)

print("Reloaded shape:", reloaded_df.shape)
print("First document ID:", reloaded_df.loc[0, "Id"])
print("First document tokens:", reloaded_df.loc[0, "Tokens"][:30])

assert len(reloaded_df) == len(processed_df)
assert reloaded_df["Id"].is_unique
assert reloaded_df["Tokens"].apply(lambda x: isinstance(x, list)).all()

print("All output validation checks passed.")

Reloaded shape: (44997, 13)
First document ID: 34552656
First document tokens: ['java', 'repeat', 'task', 'every', 'random', 'second', 'm', 'already', 'familiar', 'with', 'repeating', 'task', 'every', 'n', 'second', 'by', 'using', 'java.util.timer', 'and', 'java.util.timertask', 'but', 'let', 'say', 'want', 'to', 'print', 'hello', 'world', 'to', 'console']
All output validation checks passed.


In [141]:
import pandas as pd
import json

df = pd.read_csv("processed_documents.csv.gz", compression="gzip")
df["Tokens"] = df["Tokens"].apply(json.loads)

## 14. Handoff to Phase 3 — Inverted Index

The next stage should consume:
- `Id`: unique document identifier.
- `Tokens`: final token list for each document.
- `Title` / `Document`: original content for displaying search results.
- `Y`: optional metadata for later analysis; never treat it as a search term.

Example inverted index:

```python
{
    "python": [101, 205, 310],
    "list": [101, 205],
    "sort": [101, 310]
}
```

### Load the output in the next notebook

```python
import pandas as pd
import json

df = pd.read_csv("processed_documents.csv.gz", compression="gzip")
df["Tokens"] = df["Tokens"].apply(json.loads)
```

### Important project notes
1. Keep the original `Document`; never overwrite it with processed text.
2. Use `Id` as the document key, not the DataFrame row number.
3. Apply exactly the same preprocessing to user queries as to documents.
4. Do not remove negation words without an experiment showing that it improves retrieval.
5. Inspect programming terms and sample queries before freezing the pipeline.
6. Record changes to the tokenizer or stopword list so all team members use the same rules.

In [142]:
print(processed_df.shape)
print(processed_df[["Id", "Tokens"]].head(3))

(44997, 13)
         Id                                             Tokens
0  34552656  [java, repeat, task, every, random, second, m,...
1  34553034  [why, java, optionals, immutable, d, like, to,...
2  34553174  [text, overlay, image, with, darkened, opacity...


# Phase 3 — Inverted Index

## 1. Build the Inverted Index

In [143]:
from collections import Counter

processed_df = processed_df.sort_values("Id")

inverted_index = {}

for doc_id, tokens in zip(processed_df["Id"], processed_df["Tokens"]):
    for term, tf in Counter(tokens).items():
        if term not in inverted_index:
            inverted_index[term] = []
        inverted_index[term].append([int(doc_id), tf])

N = len(processed_df)
print("Documents:", N)
print("Vocabulary size:", len(inverted_index))

Documents: 44997
Vocabulary size: 273076


## 2. Postings Examples

In [144]:
id_to_title = dict(zip(processed_df["Id"], processed_df["Title"]))

for term in ["python", "java", "c++", "pandas","panda", "not"]:
    postings = inverted_index.get(term, [])
    print(f"term: {term}   |   appears in {len(postings)} documents")
    print("first 5 postings [doc_id, tf]:", postings[:5])
    print()

print("Real questions from the postings of 'python':")
for doc_id, tf in inverted_index["python"][:3]:
    print(doc_id, "| tf =", tf, "|", id_to_title[doc_id][:70])

term: python   |   appears in 5427 documents
first 5 postings [doc_id, tf]: [[34566113, 6], [34566806, 2], [34570584, 6], [34570992, 3], [34572553, 2]]

term: java   |   appears in 5261 documents
first 5 postings [doc_id, tf]: [[34552656, 2], [34553034, 3], [34561843, 1], [34562708, 3], [34564466, 4]]

term: c++   |   appears in 2457 documents
first 5 postings [doc_id, tf]: [[34556906, 1], [34559058, 3], [34562551, 2], [34563947, 1], [34566063, 3]]

term: pandas   |   appears in 0 documents
first 5 postings [doc_id, tf]: []

term: panda   |   appears in 486 documents
first 5 postings [doc_id, tf]: [[34586069, 4], [34597926, 2], [34771256, 3], [34789337, 2], [34794067, 2]]

term: not   |   appears in 15669 documents
first 5 postings [doc_id, tf]: [[34553318, 1], [34554026, 1], [34554871, 1], [34555603, 1], [34555643, 1]]

Real questions from the postings of 'python':
34566113 | tf = 6 | Has set been deprecated in python 2?
34566806 | tf = 2 | Why use contextlib.suppress as opposed to tr

## 3. Index Statistics

In [145]:
df_counts = {t: len(p) for t, p in inverted_index.items()}

print("Total postings:", sum(df_counts.values()))
print("Terms in only 1 document:", sum(1 for v in df_counts.values() if v == 1))

top10 = sorted(df_counts.items(), key=lambda x: -x[1])[:10]
print("\nTop 10 terms by document frequency:")
for term, count in top10:
    print(term, count)

Total postings: 2637495
Terms in only 1 document: 216733

Top 10 terms by document frequency:
to 39303
in 32387
and 28799
of 22976
for 21033
how 20787
but 19898
have 19355
can 17743
with 17707


## 4. Lookup Test

In [146]:
def lookup(term):
    return [doc_id for doc_id, tf in inverted_index.get(term, [])]

ids = lookup("python")
real = sorted(processed_df.loc[processed_df["Tokens"].apply(lambda t: "python" in t), "Id"])

print("Index result:", len(ids))
print("Matches brute force:", ids == real)

Index result: 5427
Matches brute force: True


## 5. Save the Index

In [147]:
import json

with open("inverted_index.json", "w", encoding="utf-8") as f:
    json.dump({"N": N, "index": inverted_index}, f)

processed_df[["Id", "Title", "Y"]].to_csv("doc_info.csv", index=False)

print("Saved inverted_index.json and doc_info.csv")

Saved inverted_index.json and doc_info.csv
